# Python 数据处理 — 高级工程师面试精讲

本笔记覆盖数据工程师日常最核心的数据处理库知识，重点是 **性能意识** 和 **底层原理**。

| 主题 | 出现频率 |
|------|----------|
| Pandas groupby / merge / pivot_table | 高频 |
| Pandas apply vs vectorize 性能 | 高频 |
| Pandas 内存优化 & 类型降级 | 重要 |
| NumPy broadcasting 规则 | 重要 |
| Polars vs Pandas 对比 | 重要 |
| Arrow / PyArrow 内存格式 | 高频 |

In [ ]:
# Install dependencies if needed
# !pip install pandas numpy pyarrow polars

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

try:
    import pyarrow as pa
    print(f"pyarrow {pa.__version__}")
except ImportError:
    print("pyarrow not installed")

try:
    import polars as pl
    print(f"polars  {pl.__version__}")
except ImportError:
    print("polars not installed")

---
## 1. Pandas groupby — 分组聚合

### 核心模式对比

| 方法 | 输出形状 | 典型用途 |
|------|----------|----------|
| `.agg()` | 缩减行数（每组一行） | 多指标聚合 |
| `.transform()` | 与原 DataFrame 等长 | 填充均值、组内排名 |
| `.apply()` | 灵活（任意形状） | 复杂逻辑，性能最差 |
| `.filter()` | 过滤整组 | 保留满足条件的组 |

### 面试高频考点
- `groupby` 后的 `agg` 与 `transform` 的区别
- Named aggregation（避免 MultiIndex columns）
- 多列分组 + 多指标聚合

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

# Simulate a sales dataset
n = 500
df = pd.DataFrame({
    "region":    np.random.choice(["North", "South", "East", "West"], n),
    "product":   np.random.choice(["A", "B", "C"], n),
    "salesperson": np.random.choice([f"SP{i}" for i in range(20)], n),
    "revenue":   np.random.exponential(scale=5000, size=n).round(2),
    "units":     np.random.randint(1, 50, size=n),
    "month":     np.random.randint(1, 13, size=n),
})

print(df.shape)
df.head(3)

In [ ]:
# --- agg: reduce each group to a single row ---
# Named aggregation avoids multi-level column names
region_stats = df.groupby("region").agg(
    total_revenue  = ("revenue", "sum"),
    avg_revenue    = ("revenue", "mean"),
    median_revenue = ("revenue", "median"),
    total_units    = ("units", "sum"),
    num_orders     = ("revenue", "count"),
).round(2)

print("=== Region-level aggregation ===")
print(region_stats)

print()

# --- Multi-level groupby ---
product_region = df.groupby(["region", "product"]).agg(
    revenue_sum = ("revenue", "sum"),
    unit_avg    = ("units", "mean"),
).round(2).reset_index()

print("=== Multi-level groupby (region x product) ===")
print(product_region.head(6))

In [ ]:
# --- transform: broadcast aggregation back to original shape ---
# Key: output is same length as input (for column assignment)

df["region_total"]  = df.groupby("region")["revenue"].transform("sum")
df["region_mean"]   = df.groupby("region")["revenue"].transform("mean")
df["revenue_share"] = (df["revenue"] / df["region_total"] * 100).round(4)

# Rank within each region (useful for Top-N queries)
df["rank_in_region"] = df.groupby("region")["revenue"].rank(ascending=False, method="dense")

print("Sample rows showing transform columns:")
cols = ["region", "revenue", "region_total", "revenue_share", "rank_in_region"]
print(df[cols].sort_values(["region", "rank_in_region"]).head(8).to_string(index=False))

# --- filter: keep/drop entire groups ---
# Keep only regions with total revenue > 600,000
high_revenue_df = df.groupby("region").filter(lambda g: g["revenue"].sum() > 600_000)
print(f"\nOriginal rows: {len(df)}, High-revenue regions rows: {len(high_revenue_df)}")
print("Regions kept:", high_revenue_df["region"].unique())

---
## 2. Pandas merge — 连接操作

### Join 类型

| how 参数 | SQL 等价 | 结果 |
|----------|----------|------|
| `inner` | INNER JOIN | 两表都匹配的行 |
| `left` | LEFT JOIN | 左表全部 + 右表匹配 |
| `right` | RIGHT JOIN | 右表全部 + 左表匹配 |
| `outer` | FULL OUTER JOIN | 两表全部 |
| `cross` | CROSS JOIN | 笛卡尔积 |

### 高级 merge 特性
- `indicator=True`：添加 `_merge` 列，显示行来自 left/right/both
- `merge_asof`：按最近匹配（时间序列常用）
- `validate`：检查关系是否为 1:1, 1:m, m:1, m:m

In [ ]:
import pandas as pd
import numpy as np

# Create sample dimension tables
orders = pd.DataFrame({
    "order_id":   [1, 2, 3, 4, 5],
    "customer_id":[101, 102, 101, 103, 999],  # 999 has no match
    "amount":     [250.0, 340.0, 180.0, 420.0, 90.0],
})

customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104],  # 104 has no orders
    "name":        ["Alice", "Bob", "Charlie", "Diana"],
    "tier":        ["gold", "silver", "bronze", "gold"],
})

# inner join: only matched rows
inner = pd.merge(orders, customers, on="customer_id", how="inner")
print(f"INNER: {len(inner)} rows\n{inner}\n")

# left join: all orders, NaN for unmatched customers
left = pd.merge(orders, customers, on="customer_id", how="left", indicator=True)
print(f"LEFT with indicator:\n{left}\n")

# Find orders with no matching customer (anti-join pattern)
anti_join = left[left["_merge"] == "left_only"][["order_id", "customer_id", "amount"]]
print(f"Anti-join (orders without customer):\n{anti_join}")

In [ ]:
# --- merge_asof: match on nearest key (time series join) ---
# Common use case: join trades with most recent quote

trades = pd.DataFrame({
    "time":   pd.to_datetime(["09:30:01", "09:30:05", "09:31:00", "09:32:30"]),
    "symbol": ["AAPL", "AAPL", "AAPL", "AAPL"],
    "quantity": [100, 200, 150, 300],
})

quotes = pd.DataFrame({
    "time":  pd.to_datetime(["09:30:00", "09:30:03", "09:30:45", "09:32:00"]),
    "symbol": ["AAPL", "AAPL", "AAPL", "AAPL"],
    "bid":  [150.10, 150.15, 150.20, 150.25],
    "ask":  [150.12, 150.17, 150.22, 150.27],
})

# For each trade, find the most recent quote (on or before trade time)
enriched = pd.merge_asof(
    trades.sort_values("time"),
    quotes.sort_values("time"),
    on="time",
    by="symbol",
    direction="backward"   # use most recent quote <= trade time
)
print("Trades enriched with most-recent quote:")
print(enriched.to_string(index=False))

---
## 3. pivot_table vs crosstab

| 函数 | 输入 | 典型用途 |
|------|------|----------|
| `pd.pivot_table` | DataFrame | 聚合 + 重塑，可指定 aggfunc |
| `pd.crosstab` | 列/Series | 频率统计，默认 count，可归一化 |

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
n = 300
sales = pd.DataFrame({
    "region":  np.random.choice(["North","South","East","West"], n),
    "product": np.random.choice(["A","B","C"], n),
    "quarter": np.random.choice(["Q1","Q2","Q3","Q4"], n),
    "revenue": np.random.exponential(5000, n).round(2),
    "units":   np.random.randint(1, 100, n),
})

# pivot_table: region × product, aggregating revenue
pivot = pd.pivot_table(
    sales,
    values=["revenue", "units"],
    index="region",
    columns="product",
    aggfunc={"revenue": "sum", "units": "mean"},
    margins=True,          # adds "All" row/column with grand totals
    fill_value=0,
).round(2)

print("=== pivot_table (revenue sum, units mean) ===")
print(pivot)

print()

# crosstab: frequency count + normalize
ct = pd.crosstab(
    sales["region"],
    sales["product"],
    normalize="index",   # row proportions
).round(3)

print("=== crosstab (row-normalized proportions) ===")
print(ct)

---
## 4. apply vs map vs 向量化 — 性能对比

### 性能层级（从慢到快）

```
df.apply(lambda row: ..., axis=1)   ← 最慢，逐行 Python 循环
df[col].apply(func)                 ← 慢，逐元素 Python 循环  
df[col].map(dict)                   ← 中等，用字典映射
df[col].str.method()                ← 快，向量化字符串
df[col] * scalar / np.vectorize()   ← 快，NumPy ufunc
纯 NumPy 运算 / Cython              ← 最快
```

### 核心原则
- **优先向量化**：用 pandas/numpy 内置操作，避免 Python 级别循环
- `apply(axis=1)` 每行都调用一次 Python 函数，N 行 = N 次 Python 开销
- 字符串操作用 `.str` accessor，数值用 numpy ufunc

In [ ]:
import pandas as pd
import numpy as np
import time

np.random.seed(0)
N = 200_000
df = pd.DataFrame({
    "a": np.random.randn(N),
    "b": np.random.randn(N),
    "category": np.random.choice(["low", "medium", "high"], N),
})

def time_it(label, func):
    start = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - start
    print(f"{label:<45} {elapsed*1000:7.1f} ms")
    return result

print(f"{'Method':<45} {'Time':>10}")
print("-" * 57)

# Task: compute sqrt(a^2 + b^2)
r1 = time_it("apply(axis=1) — row-wise",
    lambda: df.apply(lambda row: np.sqrt(row["a"]**2 + row["b"]**2), axis=1))

r2 = time_it("np.vectorize — element-wise",
    lambda: pd.Series(np.vectorize(lambda a, b: np.sqrt(a**2 + b**2))(df["a"], df["b"])))

r3 = time_it("Vectorized (pure numpy) ← BEST",
    lambda: np.sqrt(df["a"]**2 + df["b"]**2))

print()

# Task: string category to numeric
mapping = {"low": 0, "medium": 1, "high": 2}
time_it("apply(func) — str mapping",
    lambda: df["category"].apply(lambda x: mapping[x]))
time_it("map(dict) — str mapping ← BETTER",
    lambda: df["category"].map(mapping))

# Verify all approaches produce same result
assert np.allclose(r1.values, r2.values, atol=1e-10)
assert np.allclose(r1.values, r3.values, atol=1e-10)
print("\nAll results are numerically equivalent.")

---
## 5. Pandas 内存优化 & 类型降级

### 内存优化策略

| 技术 | 节省幅度 | 适用场景 |
|------|---------|----------|
| int64 → int8/int16/int32 | 50~87% | 整数列值域小 |
| float64 → float32 | 50% | 精度要求不高时 |
| object → category | 50~98% | 低基数字符串列 |
| object → string[pyarrow] | ~30% | pandas 2.0+ |
| 稀疏数组 | 随稀疏度 | 大量 0 或 NaN |

### 何时用 category？
- 列的唯一值数量远少于行数（cardinality < 5~10%）
- 字符串重复度高（如：状态码、地区、产品分类）
- 注意：category 类型不支持所有字符串操作

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
N = 1_000_000

# Simulate a "fat" dataframe with suboptimal dtypes
df = pd.DataFrame({
    "id":        np.arange(N, dtype=np.int64),           # could be int32
    "age":       np.random.randint(18, 90, N).astype(np.int64),   # int8 sufficient
    "score":     np.random.uniform(0, 100, N),           # float64, could be float32
    "status":    np.random.choice(["active","inactive","pending"], N),  # 3 unique values
    "country":   np.random.choice(["US","UK","DE","FR","JP"], N),       # 5 unique values
    "value":     np.random.randn(N),                     # float64, keep
})

def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1e6

print(f"Original dtypes and memory:")
print(df.dtypes)
print(f"\nTotal memory: {mem_mb(df):.1f} MB")
print(f"\nPer-column memory (MB):")
print((df.memory_usage(deep=True) / 1e6).round(2))

In [ ]:
def optimize_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Automatically downcast numeric types and convert low-cardinality strings to category."""
    df = df.copy()

    for col in df.select_dtypes(include=["int64"]).columns:
        col_min, col_max = df[col].min(), df[col].max()
        if col_min >= np.iinfo(np.int8).min  and col_max <= np.iinfo(np.int8).max:
            df[col] = df[col].astype(np.int8)
        elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
            df[col] = df[col].astype(np.int16)
        elif col_min >= np.iinfo(np.int32).min and col_max <= np.iinfo(np.int32).max:
            df[col] = df[col].astype(np.int32)

    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")

    for col in df.select_dtypes(include=["object"]).columns:
        cardinality = df[col].nunique()
        if cardinality / len(df) < 0.05:   # < 5% unique values
            df[col] = df[col].astype("category")
            print(f"  {col}: object → category ({cardinality} unique values)")

    return df

df_opt = optimize_dtypes(df)

print(f"\nOptimized dtypes:")
print(df_opt.dtypes)
print(f"\nMemory before: {mem_mb(df):.1f} MB")
print(f"Memory after:  {mem_mb(df_opt):.1f} MB")
print(f"Reduction:     {(1 - mem_mb(df_opt)/mem_mb(df))*100:.1f}%")

---
## 6. NumPy Broadcasting 规则

### Broadcasting 规则（从后向前对齐维度）

两个数组的 shape 从**最后一个维度**开始比较，满足以下条件可广播：
1. 两维度相等，或
2. 其中一个维度为 1

不足的维度前面补 1。

```
A: (3, 4)   +   B: (4,)     → B 扩展为 (1,4) → (3,4) ✓
A: (3, 1)   +   B: (1, 4)   → (3, 4) ✓
A: (3, 4)   +   B: (3,)     → B 扩展为 (1,3) ≠ (4,) ✗
A: (3, 4, 5) +  B: (4, 5)   → B 扩展为 (1,4,5) → (3,4,5) ✓
```

In [ ]:
import numpy as np

# --- Example 1: Add bias vector to a matrix (common in ML) ---
# Shape: (batch_size, features) + (features,) → (batch_size, features)
batch = np.random.randn(4, 3)   # 4 samples, 3 features
bias  = np.array([0.1, 0.2, 0.3])  # shape (3,)

result = batch + bias  # bias broadcasts to (4, 3)
print(f"batch shape: {batch.shape}, bias shape: {bias.shape}")
print(f"result shape: {result.shape}")
print(f"Bias applied:\n{result.round(3)}\n")

# --- Example 2: Row-wise normalization (z-score) ---
# Normalize each sample (row) independently
mean = batch.mean(axis=1, keepdims=True)  # shape (4, 1)
std  = batch.std(axis=1, keepdims=True)   # shape (4, 1)
z_score = (batch - mean) / std            # (4,3) - (4,1) → (4,3)
print(f"mean shape: {mean.shape} (keepdims=True is crucial)")
print(f"z_score row means (should be ~0): {z_score.mean(axis=1).round(10)}")

In [ ]:
import numpy as np

# --- Example 3: Outer product / distance matrix ---
# Compute pairwise squared distances between points
points = np.array([[0, 0], [1, 0], [0, 1], [1, 1]], dtype=float)

# points[:, np.newaxis] → (4, 1, 2)
# points[np.newaxis, :] → (1, 4, 2)
# diff                  → (4, 4, 2)  [all pairwise differences]
diff = points[:, np.newaxis] - points[np.newaxis, :]
dist_sq = (diff ** 2).sum(axis=2)  # (4, 4)

print("Pairwise squared distance matrix:")
print(dist_sq)

print()

# --- Example 4: Common broadcasting errors ---
a = np.ones((3, 4))
b = np.ones((3,))

try:
    _ = a + b  # (3,4) + (3,) → error: (1,3) vs (4,) mismatch
except ValueError as e:
    print(f"Broadcasting error: {e}")

# Fix: reshape b to (3, 1)
result = a + b[:, np.newaxis]  # (3,4) + (3,1) → (3,4)
print(f"Fixed: (3,4) + (3,1) → {result.shape}")

---
## 7. Polars vs Pandas 对比

### 核心差异

| 特性 | Pandas | Polars |
|------|--------|--------|
| 执行模型 | Eager（立即执行） | Lazy（先构建查询计划，再执行）|
| 并行性 | 单线程（大多数操作）| 多线程，自动并行 |
| 内存模型 | NumPy/Cython | Apache Arrow（列式）|
| 表达式 API | 方法链 | 表达式（`pl.col`）|
| 空值 | `NaN`（float）or `pd.NA` | `null`（一致）|
| 索引 | 有（行索引）| 无（行索引）|
| 适合场景 | 通用，生态丰富 | 大数据集，性能敏感 |

In [ ]:
try:
    import polars as pl
    import pandas as pd
    import numpy as np

    np.random.seed(42)
    N = 100_000

    # Create same dataset in both libraries
    data = {
        "id":       np.arange(N),
        "group":    np.random.choice(["A","B","C","D"], N),
        "value":    np.random.randn(N),
        "quantity": np.random.randint(1, 100, N),
    }

    # --- Pandas groupby ---
    pd_df = pd.DataFrame(data)
    pd_result = (
        pd_df
        .groupby("group")
        .agg(mean_value=("value", "mean"), total_qty=("quantity", "sum"))
        .reset_index()
    )
    print("Pandas result:")
    print(pd_result)

    print()

    # --- Polars equivalent (expression API) ---
    pl_df = pl.DataFrame(data)
    pl_result = (
        pl_df
        .group_by("group")
        .agg([
            pl.col("value").mean().alias("mean_value"),
            pl.col("quantity").sum().alias("total_qty"),
        ])
        .sort("group")
    )
    print("Polars result:")
    print(pl_result)

except ImportError:
    print("polars not installed — run: pip install polars")

In [ ]:
try:
    import polars as pl
    import numpy as np

    np.random.seed(0)
    N = 500_000

    # --- Polars Lazy API: build query plan, execute once ---
    df = pl.DataFrame({
        "user_id":  np.random.randint(1, 10000, N),
        "event":    np.random.choice(["click","view","purchase","return"], N),
        "amount":   np.random.exponential(50, N),
        "day":      np.random.randint(1, 366, N),
    })

    # Build lazy query (nothing executes yet)
    lazy_query = (
        df.lazy()                                      # switch to lazy mode
        .filter(pl.col("event") == "purchase")        # predicate pushdown
        .filter(pl.col("amount") > 10)
        .with_columns([
            (pl.col("amount") * 1.1).alias("amount_with_tax"),
        ])
        .group_by("user_id")
        .agg([
            pl.col("amount_with_tax").sum().alias("total_spend"),
            pl.col("event").count().alias("num_purchases"),
        ])
        .filter(pl.col("total_spend") > 100)
        .sort("total_spend", descending=True)
        .limit(5)
    )

    print("Lazy query plan (before execution):")
    print(lazy_query.explain())  # shows optimized plan

    print("\nTop 5 spenders:")
    result = lazy_query.collect()  # execute now
    print(result)

except ImportError:
    print("polars not installed")

---
## 8. Apache Arrow & PyArrow — 内存格式

### 为什么 Arrow 重要？

Apache Arrow 定义了**跨语言的列式内存格式**，解决了数据系统互操作的「序列化税」问题。

| 特性 | 好处 |
|------|------|
| 列式存储 | 同类型数据连续存储，CPU 缓存友好，SIMD 加速 |
| 零拷贝 | Spark ↔ pandas ↔ Polars 共享内存，无需序列化 |
| 跨语言 | Python/R/Java/C++ 使用同一内存格式 |
| 内置 null | 用位图（bitmask）表示，不占用数据空间 |

### Arrow 在数据工程中的位置
- **Parquet/ORC 读写**：PyArrow 是最快的 Parquet 引擎
- **pandas 2.0**：`pd.ArrowDtype` 作为底层存储
- **Spark Arrow 优化**：`spark.sql.execution.arrow.pyspark.enabled=true`
- **Flight**：Arrow 原生的高性能 RPC 数据传输协议

In [ ]:
try:
    import pyarrow as pa
    import pyarrow.compute as pc
    import pyarrow.parquet as pq
    import pandas as pd
    import numpy as np
    import tempfile, os

    # --- Build Arrow Table with explicit schema ---
    schema = pa.schema([
        pa.field("id",       pa.int32()),
        pa.field("name",     pa.string()),
        pa.field("revenue",  pa.float64()),
        pa.field("active",   pa.bool_()),
        pa.field("tags",     pa.list_(pa.string())),
    ])

    table = pa.table({
        "id":      pa.array([1, 2, 3, 4], type=pa.int32()),
        "name":    pa.array(["Alice", "Bob", None, "Diana"]),  # null is native
        "revenue": pa.array([1000.5, 2500.0, 750.25, 3200.0]),
        "active":  pa.array([True, True, False, True]),
        "tags":    pa.array([["vip"], ["new"], [], ["vip", "loyal"]]),
    }, schema=schema)

    print("=== PyArrow Table ===")
    print(f"Shape: {table.num_rows} rows x {table.num_columns} cols")
    print(f"Schema:\n{table.schema}")
    print(f"\nTable:\n{table}")

    print()

    # --- Arrow compute operations (SIMD-accelerated) ---
    total_revenue = pc.sum(table["revenue"]).as_py()
    active_count  = pc.sum(pc.cast(table["active"], pa.int32())).as_py()
    print(f"Total revenue: {total_revenue}")
    print(f"Active users:  {active_count}")

except ImportError:
    print("pyarrow not installed — run: pip install pyarrow")

In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pandas as pd
    import numpy as np
    import tempfile, os

    np.random.seed(0)
    N = 100_000

    # --- Zero-copy: pandas ↔ Arrow ↔ pandas ---
    pd_df = pd.DataFrame({
        "id":    np.arange(N, dtype=np.int32),
        "value": np.random.randn(N),
        "label": np.random.choice(["A","B","C"], N),
    })

    # pandas → Arrow (zero-copy for numeric columns)
    arrow_table = pa.Table.from_pandas(pd_df, preserve_index=False)
    print(f"Arrow table schema: {arrow_table.schema}")

    # Arrow → pandas (zero-copy if dtypes compatible)
    pd_roundtrip = arrow_table.to_pandas()
    print(f"Roundtrip shape: {pd_roundtrip.shape}")

    print()

    # --- Write / read Parquet with PyArrow ---
    with tempfile.NamedTemporaryFile(suffix=".parquet", delete=False) as f:
        parquet_path = f.name

    pq.write_table(
        arrow_table,
        parquet_path,
        compression="snappy",
        row_group_size=10_000,
    )

    file_size_mb = os.path.getsize(parquet_path) / 1e6
    print(f"Parquet file size: {file_size_mb:.2f} MB")
    print(f"In-memory pandas size: {pd_df.memory_usage(deep=True).sum() / 1e6:.2f} MB")

    # Read back with column pushdown (only read needed columns)
    partial = pq.read_table(parquet_path, columns=["id", "value"]).to_pandas()
    print(f"Column pushdown result: {partial.shape}")

    os.unlink(parquet_path)

except ImportError:
    print("pyarrow not installed")

---
## 复习要点

### Pandas groupby
- `agg` 缩减行数，`transform` 保持原长（适合填充、组内排名）
- Named aggregation：`agg(col_name=("src_col", func))` 避免 MultiIndex
- `apply(axis=1)` 逐行，最慢；`filter` 保留/丢弃整组

### merge
- 5 种 join 类型；`indicator=True` 显示行来源
- `merge_asof` 用于时间序列最近匹配
- Anti-join 模式：left join + 过滤 `_merge == 'left_only'`

### 性能
- 向量化 > `np.vectorize` > `.apply` on Series > `.apply(axis=1)` on DataFrame
- 字符串映射用 `.map(dict)` 而非 `.apply(lambda)`
- 数值计算优先用 NumPy ufunc

### 内存优化
- 检查 `df.dtypes` 和 `df.memory_usage(deep=True)`
- 整数降级：int64 → int8/16/32；浮点：float64 → float32
- 低基数字符串列（< 5% 唯一值）用 `category` 类型

### NumPy Broadcasting
- 从后向前对齐，维度相等或其中一个为 1 才可广播
- `keepdims=True` 保持维度，确保广播正确
- `[:, np.newaxis]` 和 `[np.newaxis, :]` 控制广播方向

### Polars
- Lazy API：先构建查询计划，支持 predicate pushdown、并行执行
- `pl.col("x").mean().alias("x_mean")` 是 Polars 的表达式风格
- 无行索引，所有操作基于列表达式

### PyArrow
- 列式内存格式，跨语言零拷贝
- `pa.table` → `to_pandas()` → `Table.from_pandas()` 互转
- Parquet 写入：指定 compression、row_group_size 优化读取性能
- `columns=` 参数实现列裁剪（column pushdown）

---
## 练习

### 练习 1 — groupby 综合

给定下面的电商订单数据，完成以下任务：

1. 计算每个 `user_id` 的：总消费、平均订单金额、最大单笔金额、订单数
2. 新增列 `user_lifetime_value`（同 user 所有订单金额之和），保持原 DataFrame 行数
3. 找出每个 `category` 中金额最高的 3 笔订单（Top-3 per group）
4. 过滤掉总订单数少于 3 的 user 的所有订单

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)
N = 1000
orders = pd.DataFrame({
    "order_id":  range(N),
    "user_id":   np.random.randint(1, 50, N),
    "category":  np.random.choice(["Electronics","Clothing","Food","Books"], N),
    "amount":    np.random.exponential(200, N).round(2),
    "date":      pd.date_range("2024-01-01", periods=N, freq="h"),
})

# TODO: 1. Per-user aggregate stats

# TODO: 2. Add user_lifetime_value column (same length as original)

# TODO: 3. Top-3 orders per category

# TODO: 4. Remove users with fewer than 3 orders


### 练习 2 — merge 与 anti-join

给定 `events` 表和 `users` 表：

1. 将 `events` 与 `users` 做 left join，为每个 event 附上用户信息
2. 找出出现在 `events` 中但不在 `users` 表中的 `user_id`（anti-join）
3. 用 `merge_asof` 将每个 event 与最近的一次 `campaigns` 记录关联（campaign 必须在 event 之前发生）
4. 用 `validate='m:1'` 确保 events.user_id → users.user_id 是多对一关系

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)

events = pd.DataFrame({
    "event_id":  range(20),
    "user_id":   np.random.choice([1,2,3,4,5,99,100], 20),  # 99,100 not in users
    "event_time": pd.date_range("2024-01-01", periods=20, freq="6h"),
    "event_type": np.random.choice(["login","purchase","view"], 20),
})

users = pd.DataFrame({
    "user_id":   [1,2,3,4,5],
    "name":      ["Alice","Bob","Charlie","Diana","Eve"],
    "tier":      ["gold","silver","bronze","gold","silver"],
})

campaigns = pd.DataFrame({
    "campaign_time": pd.to_datetime(["2024-01-01","2024-01-03","2024-01-05"]),
    "campaign_name": ["NewYear","WinterSale","FlashSale"],
})

# TODO: 1. Left join events with users

# TODO: 2. Anti-join: user_ids in events but not in users

# TODO: 3. merge_asof to find active campaign at time of each event

# TODO: 4. Validate m:1 relationship


### 练习 3 — 性能优化

下面的代码使用了低效的 `apply`，将其改写为向量化操作：

```python
df["result"] = df.apply(
    lambda row: row["price"] * row["quantity"] * (1 - row["discount"]) 
                if row["discount"] < 0.5 else row["price"] * row["quantity"] * 0.5,
    axis=1
)
```

1. 改写为向量化（`np.where` 或条件表达式）
2. 用 `%timeit` 或 `time.perf_counter` 比较两种方法在 500,000 行数据上的耗时
3. 验证两种方法结果完全一致

In [ ]:
import pandas as pd
import numpy as np
import time

np.random.seed(42)
N = 500_000
df = pd.DataFrame({
    "price":    np.random.uniform(10, 1000, N),
    "quantity": np.random.randint(1, 50, N),
    "discount": np.random.uniform(0, 0.8, N),
})

# Slow version (for reference — comment out for speed)
# start = time.perf_counter()
# df["result_slow"] = df.apply(
#     lambda row: row["price"] * row["quantity"] * (1 - row["discount"])
#                 if row["discount"] < 0.5 else row["price"] * row["quantity"] * 0.5,
#     axis=1
# )
# print(f"apply: {time.perf_counter()-start:.3f}s")

# TODO: Vectorized version using np.where
start = time.perf_counter()
# df["result_fast"] = ...
print(f"vectorized: {time.perf_counter()-start:.4f}s")

# TODO: Verify results match


### 练习 4 — 内存优化

给定一个 2GB 的宽表（模拟），实现一个 `optimize_dataframe(df)` 函数：

1. 对所有整数列自动降级到最小可用类型（int8/16/32/64）
2. 对 float64 列，如果精度允许，降级到 float32（检查 max 相对误差 < 1e-4）
3. 将基数（cardinality）低于总行数 1% 的 object 列转换为 category
4. 打印优化前后的每列内存使用对比和总节省量

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(0)
N = 500_000

df = pd.DataFrame({
    "id":         np.arange(N, dtype=np.int64),
    "age":        np.random.randint(0, 120, N).astype(np.int64),
    "day_of_year":np.random.randint(1, 366, N).astype(np.int64),
    "year":       np.random.randint(2000, 2025, N).astype(np.int64),
    "score":      np.random.uniform(0, 1, N),
    "price":      np.random.exponential(100, N),
    "status":     np.random.choice(["active","inactive","pending"], N),
    "country":    np.random.choice(["US","UK","DE","FR","JP","CN"], N),
    "product_id": np.random.randint(1, 100, N).astype(np.int64),
    "free_text":  [f"note_{i}" for i in range(N)],   # high cardinality, keep as object
})

def optimize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # TODO: Implement comprehensive dtype optimization
    pass

df_opt = optimize_dataframe(df)


### 练习 5 — NumPy Broadcasting

**任务**：

1. 给定矩阵 `X (100, 50)` 和权重向量 `w (50,)`，用 broadcasting 计算加权行和（每行的 `sum(x * w)`），不使用循环
2. 实现批量归一化（Batch Normalization）：对矩阵 `X (N, D)` 的每列减去列均值、除以列标准差
3. 计算 `points (M, 2)` 到 `centroids (K, 2)` 的所有成对欧氏距离，结果形状为 `(M, K)`，不使用循环

In [ ]:
import numpy as np

np.random.seed(42)

# TODO: 1. Weighted row sum
X = np.random.randn(100, 50)
w = np.random.rand(50)
# weighted_sum = ...
# Expected shape: (100,)

# TODO: 2. Batch normalization per column
X_bn = np.random.randn(1000, 20)
# X_normalized = ...
# Verify: X_normalized.mean(axis=0) ≈ 0, X_normalized.std(axis=0) ≈ 1

# TODO: 3. Pairwise distances
M, K = 200, 5
points    = np.random.randn(M, 2)
centroids = np.random.randn(K, 2)
# distances = ...
# Expected shape: (M, K)


### 练习 6 — PyArrow & Parquet

**任务**：模拟一个数据湖分区写入场景：

1. 用 PyArrow 创建一个包含 10 万行的 Table（字段：`event_date`, `region`, `user_id`, `amount`）
2. 按 `region` 列分区写入 Parquet 文件（`pq.write_to_dataset` 或手动分组写入）
3. 读取时使用 column pushdown（只读 `user_id` 和 `amount`）和 filter pushdown（只读 `region == 'North'`）
4. 比较全量读取 vs 分区+列裁剪读取的文件 IO 大小差异
5. 用 `pa.Table.from_pandas` 和 `to_pandas()` 演示双向转换，验证 schema 类型保留

In [ ]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
    import pandas as pd
    import numpy as np
    import tempfile
    import os

    np.random.seed(42)
    N = 100_000

    # TODO: 1. Create Arrow Table

    # TODO: 2. Write partitioned Parquet

    # TODO: 3. Read with column + filter pushdown

    # TODO: 4. Compare IO sizes

    # TODO: 5. Roundtrip pandas ↔ Arrow ↔ pandas with schema verification

    pass

except ImportError:
    print("pyarrow not installed — run: pip install pyarrow")
